# BIO タグ列から固有表現のスパンを取り出す

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/19_ner/bio_to_spans.ipynb)

NLP クックブック 19-08 ／ 固有表現抽出

モデルが出力した BIO タグ列を、「どこからどこまでが何のエンティティか」という形に戻したい。不正なタグ列にも対処したい。

In [1]:
def bio_to_spans(tags):
    """[(開始トークン位置, 終了トークン位置 (含まない), ラベル), ...] を返す。B- なしで始まる I- は新しいスパンの開始とみなす"""
    spans, start, label = [], None, None
    for i, tag in enumerate(list(tags) + ["O"]):
        prefix, _, lab = tag.partition("-")
        if start is not None and (prefix != "I" or lab != label):   # 今のスパンが終わる
            spans.append((start, i, label))
            start, label = None, None
        if prefix == "B" or (prefix == "I" and start is None):      # 新しいスパンが始まる
            start, label = i, lab
    return spans

tokens = ["山田", "太郎", "は", "京都", "大学", "を", "卒業", "し", "、", "東京", "都", "で", "働く"]
tags = ["B-PER", "I-PER", "O", "B-ORG", "I-ORG", "O", "O", "O", "O", "B-LOC", "I-LOC", "O", "O"]
for s, e, label in bio_to_spans(tags):
    print(label, "".join(tokens[s:e]))

PER 山田太郎
ORG 京都大学
LOC 東京都


In [2]:
# モデルの出力にありがちな不正な系列
bad = ["I-PER", "I-PER", "O", "B-ORG", "I-LOC", "O", "B-PER", "B-PER"]
print(bio_to_spans(bad))

def repair(tags):
    """不正な I- を B- に直して、正しい BIO 系列にする"""
    out, prev = [], "O"
    for tag in tags:
        if tag.startswith("I-") and prev[2:] != tag[2:]:
            tag = "B-" + tag[2:]
        out.append(tag)
        prev = tag
    return out

print(repair(bad))

[(0, 2, 'PER'), (3, 4, 'ORG'), (4, 5, 'LOC'), (6, 7, 'PER'), (7, 8, 'PER')]
['B-PER', 'I-PER', 'O', 'B-ORG', 'B-LOC', 'O', 'B-PER', 'B-PER']


- 不正な系列の扱い方（捨てる / 直す）で評価値が変わる。seqeval のデフォルトは、ここと同じく「I- で始まっても新しいスパンとみなす」寛容な方式。
- 根本的に防ぐには、CRF 層や、遷移制約付きのビタビ復号を使う。